# LeetCode 333 — Largest BST Subtree bằng Python

Link đề: https://leetcode.com/problems/largest-bst-subtree/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Dynamic Programming, Binary Tree, Tree DP

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def largestBSTSubtree(self, root: Optional[TreeNode]) -> int:
```

> **Lưu ý:** đây là bài **Premium**. Đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong một cây nhị phân bất kỳ, tìm **cây con** (gồm một nút và **toàn bộ hậu duệ**) là **cây tìm kiếm nhị phân (BST)** có **nhiều nút nhất**; trả về **số nút** của nó.
2. Phân biệt "cây con là BST" với "mọi nút thỏa so sánh cục bộ với hai con": BST đòi hỏi **mọi** giá trị cây con trái `<` gốc `<` **mọi** giá trị cây con phải (ở đây là **nghiêm ngặt**).
3. Đi từ **kiểm tra từng nút bằng duyệt trung thứ tự** `O(n²)` → **kiểm tra từ trên xuống với cận `(lo, hi)`, cắt tỉa khi gặp BST** → **quy hoạch động trên cây (hậu thứ tự) trả về `(là BST?, kích thước, min, max)`** `O(n)` → **mẹo gác (sentinel) `min = +∞, max = −∞`** → **hậu thứ tự lặp (không đệ quy)**.
4. Hiểu **vì sao DP từ dưới lên** hiệu quả: mỗi nút chỉ cần **thông tin tóm tắt** của hai cây con `(min, max, size, hợp lệ)`.
5. Biết kiểm tra bài cây bằng `TreeNode`, `build_tree` (mảng theo tầng, `None` cho nút rỗng) và một oracle **dịch đúng định nghĩa**.
6. Tránh các lỗi kinh điển: chỉ so sánh với hai con trực tiếp, cho phép giá trị trùng, nhầm cận của nút rỗng, và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân, tìm **cây con lớn nhất** vừa là **cây tìm kiếm nhị phân (BST)**, trong đó "lớn nhất" nghĩa là có **số nút nhiều nhất**.

Một **BST** là cây mà: các giá trị ở **cây con trái nhỏ hơn** giá trị của nút cha (gốc), các giá trị ở **cây con phải lớn hơn** giá trị của nút cha.

**Lưu ý:** một cây con phải bao gồm **tất cả hậu duệ** của nó.

**Ví dụ 1**

- Input: `root = [10,5,15,1,8,null,7]`
- Output: `3`
- Giải thích: cây con lớn nhất là BST là cây gốc `5` với hai con `1` và `8` (3 nút).

**Ví dụ 2**

- Input: `root = [4,2,7,2,3,5,null,2,null,null,null,null,null,1]`
- Output: `2`

**Ràng buộc:** số nút trong khoảng `[0, 10^4]`; `-10^4 <= Node.val <= 10^4`.

**Follow up:** làm được trong `O(n)` không?

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time

INF = float("inf")


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ mảng theo tầng (None = nút rỗng) như định dạng LeetCode."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level_order(root):
    """Ngược lại của build_tree (cắt None thừa ở cuối) để in cây gọn."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([10, 5, 15, 1, 8, None, 7], 3),
    ([4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1], 2),
]
for values, expected in examples:
    print(values, "| dựng lại:", to_level_order(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] | dựng lại: [10, 5, 15, 1, 8, None, 7] | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] | dựng lại: [4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] | kỳ vọng: 2


## 3. Kiến thức nền

### 3.1. BST và định nghĩa nghiêm ngặt

Cây `T` là BST nếu `T` rỗng, hoặc: **mọi** giá trị trong cây con trái `< T.val`, **mọi** giá trị trong cây con phải `> T.val`, và cả hai cây con cũng là BST. Chú ý "**mọi** giá trị trong cả cây con", không chỉ hai nút con trực tiếp. Giá trị trùng **không** được phép (so sánh nghiêm ngặt) — Ví dụ 2 cần điều này.
#### Code demo

In [2]:
def is_bst_bounds(node, lo=-INF, hi=INF):
    """Kiểm tra BST bằng cận (lo, hi): mọi giá trị phải nằm trong khoảng mở (lo, hi)."""
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return is_bst_bounds(node.left, lo, node.val) and is_bst_bounds(node.right, node.val, hi)


print(is_bst_bounds(build_tree([5, 3, 8, 1, 4])), is_bst_bounds(build_tree([5, 3, 8, 1, 6])), is_bst_bounds(build_tree([2, 2])))
# [5,3,8,1,6]: nút 6 nằm ở cây con TRÁI của 5 nhưng 6 > 5 nên không phải BST (dù 3 < 6 hợp lệ cục bộ)

True False False


### 3.2. Duyệt trung thứ tự (inorder)

Một cây là BST **⟺** duyệt trung thứ tự (trái, gốc, phải) cho dãy **tăng ngặt**. Cách thứ hai để kiểm tra BST.
#### Code demo

In [3]:
def inorder_values(node):
    if node is None:
        return []
    return inorder_values(node.left) + [node.val] + inorder_values(node.right)


def is_bst_inorder(node):
    vals = inorder_values(node)
    return all(vals[i] < vals[i + 1] for i in range(len(vals) - 1))


for values in ([5, 3, 8, 1, 4], [5, 3, 8, 1, 6], [2, 2]):
    t = build_tree(values)
    print(values, inorder_values(t), is_bst_inorder(t), is_bst_bounds(t))

[5, 3, 8, 1, 4] [1, 3, 4, 5, 8] True True
[5, 3, 8, 1, 6] [1, 3, 6, 5, 8] False False
[2, 2] [2, 2] False False


### 3.3. Thông tin tóm tắt của cây con: `(min, max, size, hợp lệ)`

Để biết một cây gốc `u` có là BST không, **không cần xem lại toàn bộ cây con**: chỉ cần biết với mỗi cây con của nó: **có là BST không**, **giá trị lớn nhất** (để so với `u.val`), **giá trị nhỏ nhất** (để so với `u.val` ở phía phải), và **số nút**. Từ đó `u` là BST nếu `lmax < u.val < rmin` và cả hai con là BST. Đây là **quy hoạch động trên cây** theo thứ tự **hậu thứ tự** (xử lý con trước cha).
#### Code demo

In [4]:
# Quy ước cây rỗng: là BST, size = 0, min = +inf, max = -inf  (để so sánh với mọi giá trị đều đúng)
empty_info = (True, 0, INF, -INF)
print(empty_info)
print("mọi giá trị v: lmax < v là", -INF < 5, "và v < rmin là", 5 < INF)

(True, 0, inf, -inf)
mọi giá trị v: lmax < v là True và v < rmin là True


### 3.4. Đệ quy sâu và giới hạn của Python

Cây có tới `10^4` nút có thể sâu `10^4` (xích). Mặc định Python cho đệ quy sâu ~1000. Ta dùng `sys.setrecursionlimit(...)` hoặc viết bản **lặp**. Cách lặp (hậu thứ tự dùng ngăn xếp) không phụ thuộc giới hạn này.
#### Code demo

In [5]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
sys.setrecursionlimit(100_000)
print("sau khi nâng:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000
sau khi nâng: 100000


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `[10, 5, 15, 1, 8, None, 7]`

```
        10
       /  \
      5    15
     / \     \
    1   8     7
```

- `1`, `8`, `7`: lá, mỗi lá là BST kích thước `1`.
- `5` (con `1`, `8`): `1 < 5 < 8` ⇒ BST, **kích thước 3**.
- `15` (con phải `7`): `7` ở cây con **phải** của `15` nhưng `7 < 15` ⇒ **không** phải BST.
- `10`: cây con phải chứa `7 < 10` (vi phạm), nên **không** là BST (dù `5 < 10 < 15` cục bộ).

Đáp án: `max(1, 1, 1, 3) = 3`. Quan sát: ở gốc `10` ta có `5 < 10 < 15` (cục bộ ổn) nhưng cây con phải chứa `7 < 10` nên **không** là BST. Chỉ so với hai nút con trực tiếp là **chưa đủ**: cần xét **toàn bộ** cây con, qua `max` và `min`. (Ví dụ nguy hiểm hơn: `[10, 5, 15, None, None, 6, 20]` — mọi so sánh cha–con đều đúng nhưng `6 < 10`.)

## 5. Cách 1 — Brute Force / baseline: kiểm tra mọi nút bằng duyệt trung thứ tự

**Ý tưởng.** Với **mỗi nút** `u`, coi `u` là gốc của một cây con; duyệt trung thứ tự cây con đó; nếu dãy tăng ngặt thì nó là BST, kích thước là số nút. Lấy `max`.
#### Code demo

In [6]:
def all_nodes(root):
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return out


def largest_bst_inorder(root):
    best = 0
    for u in all_nodes(root):
        vals = inorder_values(u)
        if all(vals[i] < vals[i + 1] for i in range(len(vals) - 1)):
            best = max(best, len(vals))
    return best


for values, expected in examples:
    print(values, "->", largest_bst_inorder(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] -> 3 | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] -> 2 | kỳ vọng: 2


**Độ phức tạp (Cách 1).** **Time `O(n · h)`** (mỗi nút duyệt cây con của nó; tổng kích thước các cây con là `O(n·h)`, tối đa `O(n²)` với cây xích), **Space `O(n)`**.

**Nút thắt.** Mỗi nút bị kiểm tra lại từ đầu nhiều lần (một lần cho mỗi tổ tiên); các cây con chung nhau thông tin.

## 6. Ý tưởng thuật toán / bước đột phá

**Làm từ dưới lên (hậu thứ tự).** Với mỗi nút `u`, sau khi đã biết thông tin của cây con trái `L = (isBST, size, min, max)` và cây con phải `R`:

`u` là BST ⟺ `L.isBST` **và** `R.isBST` **và** `L.max < u.val < R.min`.

Nếu đúng: `size(u) = L.size + R.size + 1`, `min(u) = min(L.min, u.val)`, `max(u) = max(R.max, u.val)`; cập nhật đáp án `best = max(best, size(u))`. Nếu sai: `u` không là BST (và mọi tổ tiên cũng không). Mỗi nút làm hằng số phép ⇒ **`O(n)`**.

## 7. Các cách tiếp theo

### Cách 2 — Từ trên xuống: kiểm tra BST bằng cận, cắt tỉa khi gặp BST

**Giải thích.** Hàm `solve(u)`: nếu cây gốc `u` là BST (kiểm tra bằng cận `(lo, hi)`) thì **trả về kích thước cả cây** (mọi cây con của nó nhỏ hơn, không cần xét); nếu không thì trả `max(solve(u.left), solve(u.right))`. Chạy tốt cho cây cân bằng; với cây lệch vẫn `O(n²)`.
#### Code demo

In [7]:
def subtree_size(node):
    if node is None:
        return 0
    return 1 + subtree_size(node.left) + subtree_size(node.right)


def largest_bst_top_down(root):
    if root is None:
        return 0
    if is_bst_bounds(root):
        return subtree_size(root)
    return max(largest_bst_top_down(root.left), largest_bst_top_down(root.right))


for values, expected in examples:
    print(values, "->", largest_bst_top_down(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] -> 3 | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 2:** **Time `O(n²)`** trong trường hợp xấu (cây lệch mà gốc/phần lớn nút không phải BST), thường `O(n log n)` trên cây cân bằng; **Space `O(h)`** (ngăn xếp đệ quy).

### Cách 3 — DP hậu thứ tự trả về `(là BST?, size, min, max)`

**Giải thích.** Như mục 6, với biến toàn cục `best`.
#### Code demo

In [8]:
def largest_bst_dp(root):
    best = 0

    def dfs(node):
        nonlocal best
        if node is None:
            return True, 0, INF, -INF                    # cây rỗng: là BST, 0 nút, min=+inf, max=-inf
        l_ok, l_size, l_min, l_max = dfs(node.left)
        r_ok, r_size, r_min, r_max = dfs(node.right)
        if l_ok and r_ok and l_max < node.val < r_min:
            size = l_size + r_size + 1
            best = max(best, size)
            return True, size, min(l_min, node.val), max(r_max, node.val)
        return False, 0, 0, 0                            # không phải BST: các trường còn lại không dùng

    dfs(root)
    return best


for values, expected in examples:
    print(values, "->", largest_bst_dp(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] -> 3 | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 3:** **Time `O(n)`** (mỗi nút thăm một lần, xử lý hằng số), **Space `O(h)`** ngăn xếp đệ quy.

### Cách 4 — Mẹo gác (sentinel): nút không phải BST trả `min = −∞, max = +∞`

**Giải thích.** Gộp cờ "là BST?" vào cặp `(min, max)`: nếu cây con **không** phải BST, ta trả `(min, max) = (−∞, +∞)` — bất kỳ phép so sánh `l_max < val` hay `val < r_min` ở nút cha **tự động thất bại**. Đồng thời trả về trường `size` = **kích thước BST lớn nhất bên trong cây con** (nếu cây con không phải BST thì là `max(L.size, R.size)`), nên đáp án là `size` của gốc. Chỉ cần một hàm trả bộ ba `(min, max, size)`.
#### Code demo

In [9]:
def largest_bst_sentinel(root):
    def dfs(node):
        if node is None:
            return INF, -INF, 0                          # (min, max, size)
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max < node.val < r_min:                      # cả hai con là BST và nút này hợp lệ
            return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
        return -INF, INF, max(l_size, r_size)            # gác: làm mọi so sánh ở cha thất bại

    return dfs(root)[2]


for values, expected in examples:
    print(values, "->", largest_bst_sentinel(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] -> 3 | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(h)`**.

Vì sao `(−∞, +∞)` đủ để "đánh dấu hỏng": ở cha, điều kiện `l_max < val` với `l_max = +∞` luôn sai, và `val < r_min` với `r_min = −∞` luôn sai. Mà nếu cây con **hợp lệ** thì `l_max < val < r_min` đúng đắn là điều kiện BST. Lưu ý: nút **rỗng** dùng `(+∞, −∞)` (ngược lại) để mọi so sánh luôn đúng.

### Cách 5 — Hậu thứ tự lặp (ngăn xếp tường minh, không đệ quy)

**Giải thích.** Dùng ngăn xếp chứa cặp `(nút, đã_thăm_con?)`. Lần đầu gặp nút: đẩy lại chính nó với cờ `True`, rồi đẩy hai con; lần thứ hai (khi cả hai con đã xong) tính thông tin của nút từ bảng kết quả `info`. Không có đệ quy nên **không giới hạn độ sâu**.
#### Code demo

In [10]:
def largest_bst_iterative(root):
    if root is None:
        return 0
    info = {}                                            # id(nút) -> (min, max, size)
    empty = (INF, -INF, 0)
    stack = [(root, False)]
    while stack:
        node, seen = stack.pop()
        if node is None:
            continue
        if not seen:
            stack.append((node, True))                   # quay lại sau khi hai con xong
            stack.append((node.right, False))
            stack.append((node.left, False))
            continue
        l_min, l_max, l_size = info[id(node.left)] if node.left else empty
        r_min, r_max, r_size = info[id(node.right)] if node.right else empty
        if l_max < node.val < r_min:
            info[id(node)] = (min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1)
        else:
            info[id(node)] = (-INF, INF, max(l_size, r_size))
    return info[id(root)][2]


for values, expected in examples:
    print(values, "->", largest_bst_iterative(build_tree(values)), "| kỳ vọng:", expected)

[10, 5, 15, 1, 8, None, 7] -> 3 | kỳ vọng: 3
[4, 2, 7, 2, 3, 5, None, 2, None, None, None, None, None, 1] -> 2 | kỳ vọng: 2


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (bảng `info` + ngăn xếp), nhưng **không đệ quy**.

> Cách 4 là bản nộp gọn nhất; Cách 5 an toàn khi cây rất sâu. Cách 3 dễ đọc nhất cho người mới.

## 8. TRACE từng bước

Ta trace Cách 3 trên ví dụ 1 theo hậu thứ tự: in từng nút khi nó **hoàn tất**, kèm thông tin con trái, con phải, kết luận và `best`.
#### Code demo

In [11]:
def trace_dp(root):
    best = 0

    def dfs(node, depth=0):
        nonlocal best
        pad = "  " * depth
        if node is None:
            return True, 0, INF, -INF
        l = dfs(node.left, depth + 1)
        r = dfs(node.right, depth + 1)
        l_ok, l_size, l_min, l_max = l
        r_ok, r_size, r_min, r_max = r
        ok = l_ok and r_ok and l_max < node.val < r_min
        if ok:
            size = l_size + r_size + 1
            best = max(best, size)
            res = (True, size, min(l_min, node.val), max(r_max, node.val))
            why = f"{l_max} < {node.val} < {r_min}"
        else:
            res = (False, 0, 0, 0)
            why = "con không BST" if not (l_ok and r_ok) else f"vi phạm: {l_max} < {node.val} < {r_min} sai"
        print(f"{pad}nút {node.val:>3}: trái(size={l_size}, max={l_max}) phải(size={r_size}, min={r_min}) -> {'BST' if ok else 'KHÔNG BST':<9} ({why}) best={best}")
        return res

    dfs(root)
    return best


trace_dp(build_tree([10, 5, 15, 1, 8, None, 7]))

    nút   1: trái(size=0, max=-inf) phải(size=0, min=inf) -> BST       (-inf < 1 < inf) best=1
    nút   8: trái(size=0, max=-inf) phải(size=0, min=inf) -> BST       (-inf < 8 < inf) best=1
  nút   5: trái(size=1, max=1) phải(size=1, min=8) -> BST       (1 < 5 < 8) best=3
    nút   7: trái(size=0, max=-inf) phải(size=0, min=inf) -> BST       (-inf < 7 < inf) best=3
  nút  15: trái(size=0, max=-inf) phải(size=1, min=7) -> KHÔNG BST (vi phạm: -inf < 15 < 7 sai) best=3
nút  10: trái(size=3, max=8) phải(size=0, min=0) -> KHÔNG BST (con không BST) best=3


3

Bảng tóm tắt (ví dụ 1):

| nút | trái `(size, max)` | phải `(size, min)` | kết luận | `best` |
|---|---|---|---|---|
| 1 | `(0, -∞)` | `(0, +∞)` | BST, size 1 | 1 |
| 8 | `(0, -∞)` | `(0, +∞)` | BST, size 1 | 1 |
| 5 | `(1, 1)` | `(1, 8)` | `1 < 5 < 8` ⇒ BST, size 3 | **3** |
| 7 | rỗng | rỗng | BST, size 1 | 3 |
| 15 | rỗng | `(1, 7)` | `-∞ < 15 < 7` sai ⇒ **không BST** | 3 |
| 10 | con phải không BST | — | **không BST** | 3 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (mẹo gác) — ngắn gọn, `O(n)`. Đúng `class Solution`, đúng `largestBSTSubtree(self, root: Optional[TreeNode]) -> int`. (Nếu lo độ sâu, thêm `sys.setrecursionlimit(...)` hoặc dùng Cách 5.)
#### Code demo

In [12]:
class Solution:
    def largestBSTSubtree(self, root: Optional[TreeNode]) -> int:
        inf = float("inf")

        def dfs(node):
            # trả về (min, max, size lớn nhất của BST trong cây con)
            if node is None:
                return inf, -inf, 0
            l_min, l_max, l_size = dfs(node.left)
            r_min, r_max, r_size = dfs(node.right)
            if l_max < node.val < r_min:               # cây con trái/phải đều là BST và nút này hợp lệ
                return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
            return -inf, inf, max(l_size, r_size)      # không phải BST: gác để cha thất bại

        return dfs(root)[2]


sol = Solution()
for values, expected in examples:
    assert sol.largestBSTSubtree(build_tree(values)) == expected, (values, expected)
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

Gọi `info(u)` là kết quả hàm với cây con gốc `u`. Ta chứng minh quy nạp theo chiều cao: **nếu cây gốc `u` là BST** thì `info(u) = (min, max, size)` đúng bằng giá trị nhỏ nhất, lớn nhất, số nút; **ngược lại** `info(u) = (−∞, +∞, s)` với `s` = số nút của BST lớn nhất nằm trong cây con của `u`.

- **Cây rỗng:** `(+∞, −∞, 0)`: là BST với `size = 0`; so sánh `l_max = −∞ < val` và `val < r_min = +∞` luôn đúng.
- **Bước quy nạp:** `u` là BST ⟺ cây con trái là BST, cây con phải là BST, `max(trái) < u.val < min(phải)`. Nếu một cây con **không** là BST thì (giả thiết quy nạp) nó trả `max = +∞` (hoặc `min = −∞`) và phép so sánh tương ứng **sai**, nên `u` không bị nhận nhầm là BST. Nếu cả hai là BST thì `l_max`, `r_min` là giá trị thật nên so sánh đúng bản chất BST.
- **Kích thước:** khi `u` là BST, `size(u) = size(trái) + size(phải) + 1`. Khi không phải, BST lớn nhất trong cây con của `u` **không chứa `u`** (vì BST chứa `u` phải là cây con gốc `u` đầy đủ), nên nằm hoàn toàn trong cây con trái hoặc phải: `max(l_size, r_size)`.
- **Đáp án:** BST lớn nhất của cả cây = `size` trả về ở gốc theo phát biểu trên.
- **Cách 3 và Cách 5:** cùng công thức; Cách 5 chỉ đổi thứ tự thăm sang ngăn xếp tường minh, vẫn đảm bảo con xử lý trước cha.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút (≤ `10^4`), `h` là chiều cao.

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra từng nút bằng inorder | `O(n·h)` (tối đa `O(n²)`) | `O(n)` |
| 2. Từ trên xuống + cận | `O(n²)` xấu nhất | `O(h)` |
| 3. DP hậu thứ tự | `O(n)` | `O(h)` |
| 4. Mẹo gác | `O(n)` | `O(h)` |
| 5. Hậu thứ tự lặp | `O(n)` | `O(n)` |

Với cây xích `n = 10^4`: Cách 1 cỡ `5·10^7` phần tử duyệt (chậm); Cách 3/4/5 chỉ `10^4` bước. `h` có thể bằng `n` (xích) nên đệ quy cần giới hạn đệ quy cao hoặc dùng Cách 5.

## 12. Bộ test

`oracle(root)` **dịch đúng định nghĩa**, độc lập với cận/gác: với mỗi nút `u`, xét cây con gốc `u`; nó là BST nếu **với mọi nút `w` trong cây con đó**, mọi giá trị trong cây con trái của `w` đều `< w.val` và mọi giá trị trong cây con phải của `w` đều `> w.val`. Chi phí `O(n³)` nên chỉ dùng cho cây nhỏ.
#### Code demo

In [13]:
def subtree_values(node):
    """Danh sách giá trị của mọi nút trong cây con (không đệ quy sâu: dùng ngăn xếp)."""
    out, stack = [], [node] if node else []
    while stack:
        x = stack.pop()
        out.append(x.val)
        if x.left:
            stack.append(x.left)
        if x.right:
            stack.append(x.right)
    return out


def oracle(root):
    best = 0
    for u in all_nodes(root):
        ok = True
        for w in all_nodes(u):
            if any(v >= w.val for v in subtree_values(w.left)) or any(v <= w.val for v in subtree_values(w.right)):
                ok = False
                break
        if ok:
            best = max(best, len(subtree_values(u)))
    return best


impls = {
    "inorder": largest_bst_inorder,
    "top_down": largest_bst_top_down,
    "dp": largest_bst_dp,
    "sentinel": largest_bst_sentinel,
    "iterative": largest_bst_iterative,
    "submit": sol.largestBSTSubtree,
}

for values, expected in examples:
    assert oracle(build_tree(values)) == expected
    for key, fn in impls.items():
        assert fn(build_tree(values)) == expected, (key, values, expected, fn(build_tree(values)))
print("PASS ví dụ của đề (cả oracle)")

PASS ví dụ của đề (cả oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ([], 0, "cây rỗng"),
    ([1], 1, "một nút"),
    ([2, 1, 3], 3, "BST đầy đủ nhỏ"),
    ([2, 2], 1, "con trái trùng giá trị (không phải BST nghiêm ngặt)"),
    ([2, None, 2], 1, "con phải trùng giá trị"),
    ([1, 1, 1], 1, "mọi giá trị bằng nhau"),
    ([3, 2, None, 1], 3, "xích trái giảm dần = BST"),
    ([1, None, 2, None, 3], 3, "xích phải tăng dần = BST"),
    ([1, 2, None, 3], 1, "xích trái tăng dần: không BST, lá 3 là BST lớn nhất (size 1)"),
    ([5, 4, 6, None, None, 3, 7], 3, "cả cây không BST (3 < 5 ở cây con phải) nhưng cây gốc 6 là BST"),
    ([10, 5, 15, 1, 8, None, 7], 3, "ví dụ 1"),
    ([10, 5, 15, None, None, 6, 20], 3, "so sánh cục bộ đều đúng nhưng 6 < 10 ở cây con phải: BST lớn nhất là cây gốc 15"),
    ([-10 ** 4, None, 10 ** 4], 2, "giá trị biên"),
    ([0, -1, 1], 3, "có số âm"),
    ([5, 1, 8, None, None, 6, 9], 5, "cả cây là BST (cây con phải gốc 8 có hai con 6 và 9)"),
]
for values, expected, label in edge_cases:
    t = build_tree(values)
    assert oracle(t) == expected, (label, "oracle", oracle(t), expected)
    for key, fn in impls.items():
        assert fn(build_tree(values)) == expected, (key, label, values, expected, fn(build_tree(values)))
    print("PASS", label)

PASS cây rỗng
PASS một nút
PASS BST đầy đủ nhỏ
PASS con trái trùng giá trị (không phải BST nghiêm ngặt)
PASS con phải trùng giá trị
PASS mọi giá trị bằng nhau
PASS xích trái giảm dần = BST
PASS xích phải tăng dần = BST
PASS xích trái tăng dần: không BST, lá 3 là BST lớn nhất (size 1)
PASS cả cây không BST (3 < 5 ở cây con phải) nhưng cây gốc 6 là BST
PASS ví dụ 1
PASS so sánh cục bộ đều đúng nhưng 6 < 10 ở cây con phải: BST lớn nhất là cây gốc 15
PASS giá trị biên
PASS có số âm
PASS cả cây là BST (cây con phải gốc 8 có hai con 6 và 9)


### 12.2. Tính chất và phản ví dụ

- Đáp án luôn `0 ≤ ans ≤ n`; cây rỗng ⇒ `0`; cây không rỗng ⇒ `ans ≥ 1` (mỗi lá là một BST).
- **Cây là BST ⟺ đáp án = số nút** (kiểm tra bằng cả `is_bst_bounds`).
- **Phản chiếu gương** (đổi trái–phải và đổi dấu giá trị) không đổi đáp án.
- **Phản ví dụ** cho kiểm tra cục bộ (chỉ so sánh với hai nút con trực tiếp): `[10, 5, 15, None, None, 6, 20]`.
#### Code demo

In [15]:
def random_tree(rng, n, value_mode):
    """Cây ngẫu nhiên n nút. value_mode: 'bst' (giá trị theo inorder tăng), 'bst_noisy' (BST rồi đổi vài giá trị), 'small' (giá trị nhỏ, có trùng)."""
    if n == 0:
        return None
    nodes = [TreeNode() for _ in range(n)]
    # dựng hình dạng ngẫu nhiên: nút i (i>=1) gắn vào một ô trống ngẫu nhiên
    slots = [(0, "left"), (0, "right")]
    for i in range(1, n):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        setattr(nodes[parent], side, nodes[i])
        slots.extend([(i, "left"), (i, "right")])
    root = nodes[0]
    order = []                                           # thứ tự inorder (lặp)
    stack, cur = [], root
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        order.append(cur)
        cur = cur.right
    if value_mode in ("bst", "bst_noisy"):
        vals = sorted(rng.sample(range(-10 ** 4, 10 ** 4 + 1), n))
        for node, v in zip(order, vals):
            node.val = v
        if value_mode == "bst_noisy":
            for _ in range(rng.randint(1, 2)):
                rng.choice(order).val = rng.randint(-10 ** 4, 10 ** 4)
    else:
        for node in order:
            node.val = rng.randint(-3, 3)
    return root


rng = random.Random(333)
for _ in range(300):
    n = rng.randint(0, 12)
    t = random_tree(rng, n, rng.choice(["bst", "bst_noisy", "small"]))
    ans = sol.largestBSTSubtree(t)
    assert 0 <= ans <= n and (n == 0 or ans >= 1)
    assert (ans == n) == is_bst_bounds(t)


def mirror_negate(node):
    if node is None:
        return None
    return TreeNode(-node.val, mirror_negate(node.right), mirror_negate(node.left))


for _ in range(300):
    t = random_tree(rng, rng.randint(0, 12), rng.choice(["bst", "bst_noisy", "small"]))
    assert sol.largestBSTSubtree(t) == sol.largestBSTSubtree(mirror_negate(t))
print("PASS: giới hạn [0, n], ans = n khi và chỉ khi là BST, đối xứng gương + đổi dấu")


def sai_so_sanh_cuc_bo(root):
    """SAI: chỉ so sánh nút với hai con trực tiếp, tưởng đó là BST."""
    def ok(node):
        if node is None:
            return True, 0
        l_ok, l_size = ok(node.left)
        r_ok, r_size = ok(node.right)
        good = l_ok and r_ok and (node.left is None or node.left.val < node.val) and (node.right is None or node.right.val > node.val)
        return good, (l_size + r_size + 1 if good else max(l_size, r_size))
    return ok(root)[1]


bad = build_tree([10, 5, 15, None, None, 6, 20])
print("so sánh cục bộ:", sai_so_sanh_cuc_bo(bad), "| đúng:", sol.largestBSTSubtree(bad))
assert sai_so_sanh_cuc_bo(bad) != sol.largestBSTSubtree(bad)

PASS: giới hạn [0, n], ans = n khi và chỉ khi là BST, đối xứng gương + đổi dấu
so sánh cục bộ: 5 | đúng: 3


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2333)
counts = {}
for trial in range(2500):
    n = rng.randint(0, 11)
    mode = rng.choice(["bst", "bst_noisy", "bst_noisy", "small"])
    t = random_tree(rng, n, mode)
    expected = oracle(t)
    counts[expected == n] = counts.get(expected == n, 0) + 1
    for key, fn in impls.items():
        assert fn(t) == expected, (key, to_level_order(t), expected, fn(t))
print(f"PASS 2500 cây ngẫu nhiên so với oracle định nghĩa (cả cây là BST: {counts.get(True, 0)}, không: {counts.get(False, 0)})")


def chain(n, direction, values):
    root = TreeNode(values[0])
    cur = root
    for v in values[1:]:
        node = TreeNode(v)
        setattr(cur, direction, node)
        cur = node
    return root


def balanced_bst(sorted_vals):
    if not sorted_vals:
        return None
    m = len(sorted_vals) // 2
    return TreeNode(sorted_vals[m], balanced_bst(sorted_vals[:m]), balanced_bst(sorted_vals[m + 1:]))


N = 10_000
vals_inc = list(range(-5000, 5000))
big_cases = [
    ("xích phải tăng dần (BST)", chain(N, "right", vals_inc), N),
    ("xích trái giảm dần (BST)", chain(N, "left", vals_inc[::-1]), N),
    ("xích phải, hỏng ở cuối", chain(N, "right", vals_inc[:-1] + [-5000]), None),
    ("BST cân bằng", balanced_bst(vals_inc), N),
    ("ngẫu nhiên giá trị nhỏ", random_tree(random.Random(1), N, "small"), None),
    ("BST có vài giá trị bị đổi", random_tree(random.Random(2), N, "bst_noisy"), None),
]
for label, t, expected in big_cases:
    row, answers = [], set()
    for key in ("sentinel", "dp", "iterative", "submit", "top_down"):
        if key == "top_down" and label.startswith("xích"):
            continue                                       # O(n^2) trên xích: bỏ qua để notebook chạy nhanh
        t0 = time.perf_counter()
        r = impls[key](t)
        dt = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{key}={dt:7.1f}ms")
    assert len(answers) == 1, (label, answers)
    ans = answers.pop()
    if expected is not None:
        assert ans == expected, (label, ans, expected)
    print(f"{label:<28} ans={ans:>6} | " + " ".join(row))
print("PASS n = 10^4")

PASS 2500 cây ngẫu nhiên so với oracle định nghĩa (cả cây là BST: 1231, không: 1269)


xích phải tăng dần (BST)     ans= 10000 | sentinel=    4.2ms dp=    3.8ms iterative=    4.8ms submit=    3.0ms


xích trái giảm dần (BST)     ans= 10000 | sentinel=    3.1ms dp=    4.0ms iterative=    5.3ms submit=    3.2ms
xích phải, hỏng ở cuối       ans=     1 | sentinel=    2.8ms dp=    2.6ms iterative=    4.2ms submit=    2.9ms
BST cân bằng                 ans= 10000 | sentinel=    2.4ms dp=    2.9ms iterative=    5.3ms submit=    2.2ms top_down=    1.6ms


ngẫu nhiên giá trị nhỏ       ans=     5 | sentinel=    3.1ms dp=    2.1ms iterative=    6.1ms submit=    2.3ms top_down=    2.4ms
BST có vài giá trị bị đổi    ans=  1946 | sentinel=    3.1ms dp=    2.9ms iterative=    5.4ms submit=    2.4ms top_down=    3.3ms
PASS n = 10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so sánh nút với hai con trực tiếp
Xem `sai_so_sanh_cuc_bo`: với `[10, 5, 15, None, None, 6, 20]` cho `5` thay vì `3`. **Vì sao sai:** BST đòi hỏi **mọi** giá trị cây con trái `<` gốc `<` mọi giá trị cây con phải. Ở đây `6` nằm ở cây con phải của `10` nhưng `6 < 10`, dù `6 < 15` (so sánh cha–con đúng). **Code đúng:** mang theo `(min, max)` của cả cây con.

### Lỗi 2: Cho phép giá trị trùng (`<=`)
#### Code sai

In [17]:
def sai_cho_phep_trung(root):
    def dfs(node):
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max <= node.val <= r_min:                      # SAI: phải là nghiêm ngặt <
            return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
        return -INF, INF, max(l_size, r_size)
    return dfs(root)[2]


ex2 = build_tree(examples[1][0])
print("sai:", sai_cho_phep_trung(ex2), "| đúng:", sol.largestBSTSubtree(ex2))
assert sai_cho_phep_trung(ex2) != sol.largestBSTSubtree(ex2)

sai: 8 | đúng: 2


**Vì sao sai:** đề yêu cầu cây con trái **nhỏ hơn** và cây con phải **lớn hơn** (nghiêm ngặt); giá trị trùng làm hỏng BST. Ví dụ 2 cần điều này. **Code đúng:** `l_max < node.val < r_min`.

### Lỗi 3: Chỉ kiểm tra gốc có phải BST không
#### Code sai

In [18]:
def sai_chi_xet_goc(root):
    return subtree_size(root) if is_bst_bounds(root) else 0          # SAI: BST lớn nhất có thể nằm sâu bên trong


print("sai:", sai_chi_xet_goc(bad), "| đúng:", sol.largestBSTSubtree(bad))
assert sai_chi_xet_goc(bad) != sol.largestBSTSubtree(bad)

sai: 0 | đúng: 3


**Vì sao sai:** cây con BST lớn nhất thường không phải là cả cây. **Code đúng:** xét **mọi** nút làm gốc (qua DP hậu thứ tự), lấy `max`.

### Lỗi 4: Chọn sai cận cho nút rỗng
#### Code sai

In [19]:
def sai_can_nut_rong(root):
    def dfs(node):
        if node is None:
            return 0, 0, 0                                      # SAI: (min, max) = (0, 0) thay vì (+inf, -inf)
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max < node.val < r_min:
            return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
        return -INF, INF, max(l_size, r_size)
    return dfs(root)[2]


for values in ([5], [2, 1, 3], [-5, -7, -3]):
    t = build_tree(values)
    print(values, "sai:", sai_can_nut_rong(t), "| đúng:", sol.largestBSTSubtree(t))
assert sai_can_nut_rong(build_tree([5])) != sol.largestBSTSubtree(build_tree([5]))

[5] sai: 0 | đúng: 1
[2, 1, 3] sai: 0 | đúng: 3
[-5, -7, -3] sai: 0 | đúng: 3


**Vì sao sai:** nút rỗng phải "không ràng buộc": `max = −∞` (mọi giá trị đều lớn hơn) và `min = +∞` (mọi giá trị đều nhỏ hơn). Với `(0, 0)` thì lá `5` gặp `val < r_min = 0` sai ⇒ bị loại, hoặc giá trị âm bị so sánh sai. **Code đúng:** `(+∞, −∞, 0)`.

### Lỗi 5: Cộng sai kích thước (lấy `max + 1` thay vì tổng)
#### Code sai

In [20]:
def sai_kich_thuoc(root):
    def dfs(node):
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max < node.val < r_min:
            return min(l_min, node.val), max(r_max, node.val), max(l_size, r_size) + 1     # SAI: phải là tổng
        return -INF, INF, max(l_size, r_size)
    return dfs(root)[2]


t = build_tree([2, 1, 3])
print("sai:", sai_kich_thuoc(t), "| đúng:", sol.largestBSTSubtree(t))
assert sai_kich_thuoc(t) != sol.largestBSTSubtree(t)

sai: 2 | đúng: 3


**Vì sao sai:** số nút của một cây là `size(trái) + size(phải) + 1`, không phải `max(...) + 1`. **Code đúng:** `l_size + r_size + 1`.

### Lỗi 6 (hiệu năng): Đệ quy quá sâu
#### Code sai

In [21]:
def goi_voi_gioi_han(fn, root, limit):
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(limit)
    try:
        return fn(root)
    except RecursionError:
        return "RecursionError"
    finally:
        sys.setrecursionlimit(old)


deep = chain(5000, "right", list(range(5000)))
print("đệ quy, giới hạn 1000:", goi_voi_gioi_han(sol.largestBSTSubtree, deep, 1000))
print("lặp, giới hạn 1000   :", goi_voi_gioi_han(largest_bst_iterative, deep, 1000))
print("đệ quy, giới hạn 100000:", goi_voi_gioi_han(sol.largestBSTSubtree, deep, 100_000))
assert goi_voi_gioi_han(sol.largestBSTSubtree, deep, 1000) == "RecursionError"
assert goi_voi_gioi_han(largest_bst_iterative, deep, 1000) == 5000

đệ quy, giới hạn 1000: RecursionError
lặp, giới hạn 1000   : 5000
đệ quy, giới hạn 100000: 5000


**Vì sao nguy hiểm:** cây xích sâu `10^4` làm hàm đệ quy vượt giới hạn mặc định. **Code đúng:** nâng `sys.setrecursionlimit` hoặc dùng bản lặp (Cách 5).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**cây con lớn nhất thỏa tính chất X**" và X xác định được từ **tóm tắt cây con** → **DP hậu thứ tự** trả bộ giá trị tóm tắt.
- BST ⟺ `max(trái) < val < min(phải)` **và hai con đều BST**; cây rỗng: `min = +∞`, `max = −∞`.
- **Mẹo gác** `(−∞, +∞)` cho cây con hỏng giúp bỏ cờ riêng, trả luôn `max size` lên trên.
- Nhớ **nghiêm ngặt** (`<`, không `<=`).
- Cây sâu: bản **lặp** hoặc nâng giới hạn đệ quy.
- Họ hàng: **Validate Binary Search Tree** (98), **Recover BST** (99), **Binary Tree Maximum Path Sum** (124), **Count Univalue Subtrees** (250), **Diameter of Binary Tree** (543), **Maximum Sum BST in Binary Tree** (1373 — gần như y hệt, đổi `size` thành tổng giá trị).
- Thử: `[]`, `[2,2]`, `[10,5,15,None,None,6,20]` (so sánh cục bộ sai), xích sâu.

## 15. Biến thể của bài

1. **Maximum Sum BST in Binary Tree** (1373): thay `size` bằng **tổng giá trị**, tối đa hóa tổng (cho phép giá trị âm → đáp án không âm).
2. **Trả về gốc** của BST lớn nhất (thay vì kích thước).
3. **Kiểm tra cả cây có là BST** (98) — chỉ cần `largest == n`.
4. **Đếm số cây con là BST**.
5. **BST lớn nhất cho phép trùng giá trị** (`<=` ở một phía).
#### Code demo

In [22]:
def max_sum_bst(root):
    """LeetCode 1373: tổng lớn nhất của một cây con là BST (>= 0)."""
    best = 0

    def dfs(node):
        nonlocal best
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_sum = dfs(node.left)
        r_min, r_max, r_sum = dfs(node.right)
        if l_max < node.val < r_min:
            total = l_sum + r_sum + node.val
            best = max(best, total)
            return min(l_min, node.val), max(r_max, node.val), total
        return -INF, INF, 0

    dfs(root)
    return best


def count_bst_subtrees(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return INF, -INF, True
        l_min, l_max, l_ok = dfs(node.left)
        r_min, r_max, r_ok = dfs(node.right)
        if l_ok and r_ok and l_max < node.val < r_min:
            count += 1
            return min(l_min, node.val), max(r_max, node.val), True
        return -INF, INF, False

    dfs(root)
    return count


print(max_sum_bst(build_tree([1, 4, 3, 2, 4, 2, 5, None, None, None, None, None, None, 4, 6])))   # 20
print(count_bst_subtrees(build_tree([10, 5, 15, 1, 8, None, 7])), count_bst_subtrees(build_tree([2, 1, 3])))   # 4 và 3

20
4 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Inorder từng nút | mỗi nút duyệt cây con, kiểm tra tăng ngặt | `O(n·h)` | `O(n)` | baseline, dễ viết |
| 2. Từ trên xuống + cận | nếu cả cây con là BST thì trả size, không thì đệ quy hai con | `O(n²)` xấu nhất | `O(h)` | cây cân bằng |
| 3. DP hậu thứ tự | `(là BST?, size, min, max)` | `O(n)` | `O(h)` | dễ đọc, follow-up |
| 4. Mẹo gác | `(min, max, size)`, hỏng = `(−∞, +∞)` | `O(n)` | `O(h)` | **bản nộp gọn nhất** |
| 5. Hậu thứ tự lặp | ngăn xếp tường minh | `O(n)` | `O(n)` | cây rất sâu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán kết quả `sol.largestBSTSubtree` cho: (a) `[]`; (b) `[1]`; (c) `[2, 1, 3]`; (d) `[2, 2]`.

**Bài 2 (tính bằng tay).** Với cây `[5, 4, 6, None, None, 3, 7]`, ghi `(min, max, size)` hoặc `hỏng` cho từng nút theo hậu thứ tự và kết luận đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `largest_bst_todo(root)` theo Cách 4.
#### Code demo

In [23]:
def largest_bst_todo(root):
    # TODO 1: viết dfs(node) trả (min, max, size); nút rỗng -> (INF, -INF, 0)
    # TODO 2: nếu l_max < node.val < r_min: trả (min(l_min, val), max(r_max, val), l_size + r_size + 1)
    # TODO 3: ngược lại trả (-INF, INF, max(l_size, r_size))
    # TODO 4: return dfs(root)[2]
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples: assert largest_bst_todo(build_tree(values)) == expected

**Bài 4 (tự viết lại).** Viết `largest_bst_root(root)` trả **nút gốc** của BST lớn nhất (nếu hòa, trả nút gặp trước theo hậu thứ tự); cây rỗng trả `None`.

**Bài 5 (biến thể / mini challenge).** Viết `largest_bst_allow_dup_left(root)`: BST cho phép cây con trái `<=` gốc (trùng giá trị chỉ được ở bên trái), cây con phải vẫn `>` gốc. Thử `[2, 2]` → `2` và `[2, None, 2]` → `1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: lá luôn là BST; trùng giá trị không hợp lệ. Bài 2: nút `6` có hai lá `3`, `7`. Bài 4: đồng thời trả nút gốc ứng viên. Bài 5: đổi so sánh phía trái thành `<=`.
- **Mức 2 (vừa):** Bài 1: `0`, `1`, `3`, `1`. Bài 2: `4`: `(4,4,1)`; `3`: `(3,3,1)`; `7`: `(7,7,1)`; `6`: `3 < 6 < 7` ⇒ `(3,7,3)`; `5`: `l_max = 4 < 5 < r_min = 3` sai ⇒ hỏng; đáp án `3`. Bài 3: khung Cách 4.
- **Mức 3 (sâu):** Bài 4: hàm đệ quy trả `(min, max, size, best_size, best_node)`, hoặc giữ biến ngoài `best_size`/`best_node` và cập nhật khi `size > best_size`. Bài 5: điều kiện là `l_max <= val < r_min`; với `[2,2]`: con trái `2 <= 2` hợp lệ nên `2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert [sol.largestBSTSubtree(build_tree(v)) for v in ([], [1], [2, 1, 3], [2, 2])] == [0, 1, 3, 1]

# Bài 2
assert sol.largestBSTSubtree(build_tree([5, 4, 6, None, None, 3, 7])) == 3


# Bài 3
def largest_bst_todo_ans(root):
    def dfs(node):
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max < node.val < r_min:
            return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
        return -INF, INF, max(l_size, r_size)

    return dfs(root)[2]


for values, expected in examples:
    assert largest_bst_todo_ans(build_tree(values)) == expected


# Bài 4
def largest_bst_root(root):
    best_size, best_node = 0, None

    def dfs(node):
        nonlocal best_size, best_node
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max < node.val < r_min:
            size = l_size + r_size + 1
            if size > best_size:
                best_size, best_node = size, node
            return min(l_min, node.val), max(r_max, node.val), size
        return -INF, INF, max(l_size, r_size)

    dfs(root)
    return best_node


r = largest_bst_root(build_tree([10, 5, 15, 1, 8, None, 7]))
assert r.val == 5 and subtree_size(r) == 3
assert largest_bst_root(None) is None


# Bài 5
def largest_bst_allow_dup_left(root):
    def dfs(node):
        if node is None:
            return INF, -INF, 0
        l_min, l_max, l_size = dfs(node.left)
        r_min, r_max, r_size = dfs(node.right)
        if l_max <= node.val < r_min:
            return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
        return -INF, INF, max(l_size, r_size)

    return dfs(root)[2]


assert largest_bst_allow_dup_left(build_tree([2, 2])) == 2
assert largest_bst_allow_dup_left(build_tree([2, None, 2])) == 1
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (DP hậu thứ tự với mẹo gác)

```python
class Solution:
    def largestBSTSubtree(self, root: Optional[TreeNode]) -> int:
        inf = float("inf")

        def dfs(node):
            if node is None:
                return inf, -inf, 0                    # (min, max, size)
            l_min, l_max, l_size = dfs(node.left)
            r_min, r_max, r_size = dfs(node.right)
            if l_max < node.val < r_min:
                return min(l_min, node.val), max(r_max, node.val), l_size + r_size + 1
            return -inf, inf, max(l_size, r_size)

        return dfs(root)[2]
```

**Checklist**

- [ ] So sánh với `max` của cây con trái và `min` của cây con phải (không chỉ hai con trực tiếp)?
- [ ] Dùng so sánh **nghiêm ngặt** `<` (không trùng giá trị)?
- [ ] Cây rỗng `(+∞, −∞, 0)`; cây con hỏng `(−∞, +∞, max size)`?
- [ ] `size = l_size + r_size + 1` khi hợp lệ?
- [ ] Trả `size` ở **gốc** (hoặc cập nhật `best` ở mỗi nút BST)?
- [ ] Cây sâu `10^4`: đã tính tới độ sâu đệ quy hoặc dùng bản lặp?
- [ ] Đã thử `[]`, `[2,2]`, `[10,5,15,None,None,6,20]` (so sánh cục bộ sai), xích phải/trái?
- [ ] Skeleton đúng `class Solution` / `def largestBSTSubtree(self, root: Optional[TreeNode]) -> int`?